# Response-locked CSD plots — Flow paper

Castellar replication figures:
- **Fig5A**: response-locked CSD waveform, FCz and frontal cluster, High vs Low Flow
- **Fig5B**: response-locked CSD topomaps, 5 time bins 0–360ms, High vs Low Flow rows

Windows follow Castellar et al.:
- Pre-response: [-100, 0ms] (positive peak)
- Post-response: [0, 300ms] (negative peak)
- Topomap bins: 5 × 72ms from 0 to 360ms

Last edit: 16.10.2026 @anguyen

In [1]:
from datetime import datetime
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import mne
from mne import grand_average
from mne.defaults import DEFAULTS
from mne.viz import set_browser_backend

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from config_flow import STYLES
from participants import get_participants
from utils import load_pkl, show_latest_pkls

%matplotlib qt
set_browser_backend('qt')

plt.rcParams.update({
    'font.size':        12,
    'axes.titlesize':   13,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.titlesize': 16,
})

Using qt as 2D backend.


In [11]:
# ==============================================================================
# Configuration
# ==============================================================================

PAPER    = 'flow'
PARADIGM = 'active'
CSD      = True
BASELINE_CORR = 'bc-200-0'
label_csd     = 'CSD'
DATE_STR      = datetime.today().strftime('%Y%m%d')

if sys.platform == 'win32':
    root = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
else:
    root = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')

input_path = root / 'outputs' / 'grand_averages' / PAPER / 'raw'
root_img   = root / 'outputs' / 'img' / PAPER / "raw"
root_img.mkdir(parents=True, exist_ok=True)

n           = len(get_participants(PAPER, PARADIGM))
DATE_SOURCE = show_latest_pkls(root, PAPER, PARADIGM, 'resp', 'raw', BASELINE_CORR, None)

print(f'Input : {input_path}')
print(f'Output: {root_img}')


Latest matching files in raw/:
  [20261015] 20261015_active_36p_cond-highflow_resp_bc-200-0_CSD_raw.pkl
  [20261015] 20261015_active_36p_cond-lowflow_resp_bc-200-0_CSD_raw.pkl
  [20261014] 20261014_active_36p_cond-highflow_resp_bc-200-0_CSD_raw.pkl

→ Most recent: 20261015
Input : L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2\outputs\grand_averages\flow\raw
Output: L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2\outputs\img\flow\raw


In [12]:
# ==============================================================================
# Load pkls and compute grand averages
# ==============================================================================

CONDITIONS = {'High Flow': 'highflow', 'Low Flow': 'lowflow'}

evokeds_raw: dict[str, list] = {}
ga:          dict[str, object] = {}

for plot_label, cond_key in CONDITIONS.items():
    fname = f'{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_key}_resp_{BASELINE_CORR}_{label_csd}_raw.pkl'
    pkl   = load_pkl(input_path / fname)
    key   = f'{cond_key}_response'
    evokeds_raw[plot_label] = pkl[key]
    ga[plot_label]          = grand_average(pkl[key])
    print(f'Loaded [{plot_label}] <- {fname}  (n={len(pkl[key])})')

Identifying common channels ...
Loaded [High Flow] <- 20261015_active_36p_cond-highflow_resp_bc-200-0_CSD_raw.pkl  (n=36)
Identifying common channels ...
Loaded [Low Flow] <- 20261015_active_36p_cond-lowflow_resp_bc-200-0_CSD_raw.pkl  (n=36)


## Fig5A — Waveform (FCz and frontal cluster)

In [13]:
def plot_resp_waveform(picks, cluster_label, ylim=dict(csd=[-6, 2]), save=True):
    """Response-locked CSD waveform, High vs Low Flow."""
    fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))

    mne.viz.plot_compare_evokeds(
        {label: evokeds_raw[label] for label in CONDITIONS},
        picks=picks,
        styles=STYLES,
        combine='mean',
        vlines=[],
        truncate_yaxis=False,
        truncate_xaxis=False,
        invert_y=True,
        ylim=ylim,
        show_sensors=False,
        axes=ax,
        title='',
        time_unit='ms',
    )

    # # CSD unit scaling: mV/m² -> µV/m²
    # ax.yaxis.set_major_formatter(
    #     ticker.FuncFormatter(lambda x, pos: f'{x*1000:.0f}')
    # )
    # ax.set_ylabel('Current Source Density (µV/m²)')
    ax.set_ylabel('Current Source Density (mV/m\u00b2)')

    ax.axvline(0,   color='black', linestyle='solid',  linewidth=1.2)
    ax.axhline(0,   color='black', linestyle='solid',  linewidth=1.2)
    ax.axvline(360, color='black', linestyle='dashed', linewidth=1.0)

    ax.axvspan(-100, 0,   facecolor='lightblue', alpha=0.4)
    ax.axvspan(0,    300, facecolor='#ffbd66',   alpha=0.4)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.3)
    ax.legend(loc='lower right')

    plt.subplots_adjust(left=0.05, right=0.95)
    fig.suptitle(f'[{PARADIGM.capitalize()}] Response-locked CSD')
    fig.text(
        0.5, 0.895,
        f'CSD | {cluster_label} | N={n}',
        ha='center', fontsize=12, style='italic', color='gray'
    )

    if save:
        picks_str = cluster_label.replace(' ', '_').replace(',', '')
        fname = root_img / f'{PARADIGM}_{n}p_flow_resp_{BASELINE_CORR}_{label_csd}_wave_{picks_str}_{DATE_STR}.svg'
        plt.savefig(fname, dpi=300, bbox_inches='tight')
        print(f'Saved -> {fname.name}')

    plt.show()

print('plot_resp_waveform() ready.')

plot_resp_waveform() ready.


In [14]:
# FCz only
plot_resp_waveform(
    picks         = ['FCz'],
    cluster_label = 'FCz',
    ylim          = dict(csd=[-6, 2]),
)

combining channels using "mean"
combining channels using "mean"


C:\Users\anguyen\AppData\Local\Temp\ipykernel_25100\132019459.py:5: RuntimeWarning: Only 1 channel in "picks"; cannot combine by method "mean".
  mne.viz.plot_compare_evokeds(


Saved -> active_36p_flow_resp_bc-200-0_CSD_wave_FCz_20261016.svg


In [6]:
# # Frontal cluster
# plot_resp_waveform(
#     picks         = ['F1', 'Fz', 'F2', 'FC1', 'FCz', 'FC2'],
#     cluster_label = 'Frontal cluster',
#     ylim          = dict(csd=[-3.5, 1]),
# )

combining channels using "mean"
combining channels using "mean"
Saved -> active_36p_flow_resp_bc-200-0_CSD_wave_Frontal_cluster_20261016.svg


## Fig5B — Topomaps (5 bins, 0–360ms)

In [16]:
# 5 time bins of 72ms from 0 to 360ms (following Castellar et al.)
start_ms, end_ms, bin_ms = 0, 360, 72
edges  = np.arange(start_ms, end_ms + bin_ms, bin_ms)
# times  = [(edges[i] + edges[i+1]) / 2 / 1000 for i in range(len(edges)-1)]
# average = [(edges[i+1] - edges[i]) / 1000 for i in range(len(edges)-1)]
# 5 bins of 72ms from 0 to 360ms — centers as in Castellar et al.
times   = [t/1000 for t in [36, 108, 180, 252, 324]]
average = [0.072] * 5
print(f'Bin centers (s): {[round(t,4) for t in times]}')

fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))

fig_high = ga['High Flow'].plot_topomap(
    times=times, average=average,
    axes=axes[0], colorbar=False,
    time_unit='ms', show=False,
    # scalings=dict(csd=1e6),
)
fig_low = ga['Low Flow'].plot_topomap(
    times=times, average=average,
    axes=axes[1], colorbar=False,
    time_unit='ms', show=False,
    # scalings=dict(csd=1e6),
)

plt.subplots_adjust(hspace=0.1)

# global colour scale
def get_images(fig):
    return [ax.images[0] for ax in fig.axes if ax.images]

all_images = get_images(fig_high) + get_images(fig_low)
vmin = min(im.get_clim()[0] for im in all_images)
vmax = max(im.get_clim()[1] for im in all_images)
for im in all_images:
    im.set_clim(vmin, vmax)

ch_type = mne.channel_type(ga['High Flow'].info, 0)
unit    = DEFAULTS['units'][ch_type]

cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.03])
cbar    = fig.colorbar(all_images[0], cax=cbar_ax, orientation='horizontal')
# cbar.set_label(f'CSD ({unit})')
cbar.set_label(f'CSD (mV/m\u00b2)')

fig.text(0.93, 0.70, 'High Flow',          fontsize=12, fontweight='bold', rotation=0)
fig.text(0.93, 0.28, 'Low Flow',           fontsize=12, fontweight='bold', rotation=0)

for ax in axes[1]:
    ax.set_title('')

fname_topo = root_img / f'{PARADIGM}_{n}p_flow_resp_{BASELINE_CORR}_{label_csd}_topo_5bins_{DATE_STR}.svg'
plt.savefig(fname_topo, dpi=300, bbox_inches='tight')
print(f'Saved -> {fname_topo.name}')
plt.show()

Bin centers (s): [0.036, 0.108, 0.18, 0.252, 0.324]
Saved -> active_36p_flow_resp_bc-200-0_CSD_topo_5bins_20261016.svg
